[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/tree_basics.ipynb)

# Decision trees, in pictures

Run each cell with **Shift and Enter**. Nothing has to be typed from scratch:
every task is one number to change, and the answer to each one is written
underneath it.

## 1. Flowers, measured twice

The iris table ships inside scikit-learn. Every row is one flower, and the two
columns below are the length and the width of its petal, in centimetres. The
label says which of three kinds it is.

In [ ]:
from sklearn.datasets import load_iris
import numpy as np
import matplotlib.pyplot as plt

iris = load_iris()
cols = [2, 3]
X = iris.data[:, cols]
y = iris.target
labels = [iris.feature_names[c] for c in cols]
names = list(iris.target_names)

print("rows and columns:", X.shape)
print("kinds:", {str(n): int((y == k).sum()) for k, n in enumerate(names)})

fig, ax = plt.subplots(figsize=(5.2, 3.4))
for k, mark in enumerate("os^"):
    ax.scatter(X[y == k, 0], X[y == k, 1], marker=mark, s=26,
               color=["#1e3a5f", "#b45309", "#64748b"][k], label=names[k])
ax.set_xlabel(labels[0])
ax.set_ylabel(labels[1])
ax.legend(frameon=False, fontsize=8)
plt.show()

Three kinds, fifty flowers each. One kind sits on its own in the bottom left
corner. The other two touch along one edge, and that edge is where every
mistake in this notebook happens.

**Task 1.** Change `cols = [2, 3]` to `cols = [0, 1]` and run the cell again.
Those are the sepal measurements.

*Answer.* Setosa still stands apart, and the other two kinds overlap so much
that no flat cut separates them. Petal length and petal width are the pair
worth keeping, so change it back to `cols = [2, 3]` before you go on.

## 2. One question, one cut

A decision tree asks one measurement one question. A tree of depth 1 asks a
single question and then answers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def show(model, X, y, title):
    """The two measurements, the classes, and the boxes the model cuts."""
    step = 0.02
    gx, gy = np.meshgrid(np.arange(X[:, 0].min() - 0.4, X[:, 0].max() + 0.4, step),
                         np.arange(X[:, 1].min() - 0.4, X[:, 1].max() + 0.4, step))
    zone = model.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)

    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    ax.contourf(gx, gy, zone, levels=np.arange(-0.5, len(names) + 0.5, 1.0),
                colors=["#dfe4ec", "#fbeddc", "#e9e9f4"][:len(names)])
    ax.contour(gx, gy, zone, levels=np.arange(0.5, len(names) - 0.5, 1.0),
               colors="#b45309", linewidths=1.0)
    for k in range(len(names)):
        ax.scatter(X[y == k, 0], X[y == k, 1], marker="os^"[k], s=16,
                   color=["#1e3a5f", "#b45309", "#64748b"][k],
                   label=names[k], zorder=3)
    ax.set_xlabel(labels[0])
    ax.set_ylabel(labels[1])
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8, loc="upper right")
    plt.show()

from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X, y)

column = labels[tree.tree_.feature[0]]
print("the question it asks: is %s below %.2f ?"
      % (column, tree.tree_.threshold[0]))
print("it gets right:", round(tree.score(X, y), 3))

show(tree, X, y, "one question")

The picture is cut in two by a straight line, and the line is flat because the
question is about one measurement. Everything on one side gets one answer.

**Task 2.** Change `max_depth=1` to `max_depth=2` and run the cell again.

*Answer.* A second question appears and the picture gains a second cut, at a
right angle to the first. The score goes from 0.667 to 0.96: two questions tell
the three kinds apart almost perfectly.

## 3. The same tree, drawn as a tree

Each box is a question. Take the left branch when the answer is yes.

In [ ]:
from sklearn.tree import plot_tree

deep = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
plot_tree(deep, feature_names=labels, class_names=names, filled=True,
          impurity=False, fontsize=8, ax=ax)
plt.show()

`samples` is how many flowers reached that box. `value` is how many of each
kind are in it. The colour is the kind the box answers with.

**Task 3.** A flower has petal length 5.0 and petal width 1.8. Follow it down
the tree by hand, then check yourself with the cell below.

*Answer.* Petal length 5.0 is above the first threshold, so it goes right;
petal width 1.8 is above the second, so it goes right again and lands in the
box that answers virginica.

In [ ]:
flower = [[5.0, 1.8]]
print("the tree says:", names[deep.predict(flower)[0]])

## 4. What depth does to the score

A deeper tree asks more questions. Ask enough of them and it can name every
flower it was shown, which says nothing about a flower it has not seen. Hold
some rows back and score both.

In [ ]:
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0)

depths = range(1, 11)
learned = [DecisionTreeClassifier(max_depth=d, random_state=0)
           .fit(Xtr, ytr).score(Xtr, ytr) for d in depths]
held = [DecisionTreeClassifier(max_depth=d, random_state=0)
        .fit(Xtr, ytr).score(Xte, yte) for d in depths]

fig, ax = plt.subplots(figsize=(5.2, 3.0))
ax.plot(list(depths), learned, "--", color="#64748b", label="rows it learned from")
ax.plot(list(depths), held, color="#1e3a5f", label="rows held back")
ax.set_xlabel("depth allowed")
ax.set_ylabel("share right")
ax.legend(frameon=False, fontsize=8)
plt.show()

for d, a, b in zip(depths, learned, held):
    print("depth %2d   learned %.3f   held back %.3f" % (d, a, b))

The dashed line climbs to 0.989 and stops there: past that depth the tree is
naming flowers it has already seen. The solid line is the one that matters.

**Task 4.** Read the two columns and pick the depth you would use.

*Answer.* Depth 3, at 0.950 on the rows held back, which is the best of the
ten. Every deeper tree scores 0.883 on those same rows: the extra questions
help only on flowers the tree has already been shown.

## 5. A boundary that runs at an angle

A tree cuts flat lines, one measurement at a time. When the real boundary runs
at an angle, flat cuts have to approximate it in steps.

In [ ]:
rng = np.random.default_rng(0)
P = rng.random((300, 2))
q = (P[:, 1] > P[:, 0]).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
for ax, depth in zip(axes, [2, 5]):
    fit = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(P, q)
    gx, gy = np.meshgrid(np.linspace(0, 1, 300), np.linspace(0, 1, 300))
    zone = fit.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, zone, levels=[-0.5, 0.5, 1.5],
                colors=["#dfe4ec", "#fbeddc"])
    ax.plot([0, 1], [0, 1], color="#b45309", lw=1.5)
    ax.set_title("depth %d" % depth)
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

The orange line is the boundary the data actually has. The shaded steps are
what the tree can draw with flat cuts.

**Task 5.** In the cell above, change `[2, 5]` to `[2, 12]` and run it again.

*Answer.* The right hand picture gains many more steps and hugs the orange line
much more closely. A tree can come as near as you like to a slanted boundary,
and it pays for it in cuts.

## What you can say now

A decision tree asks one measurement one question at a time, and its answer is
a chain of those questions anybody can read. Each question cuts the picture
with a flat line. More depth means more questions, a better score on the rows
it learned from, and, past a point, nothing on the rows held back.

The notebook next door does the same for a forest of these trees.